###**Programación Concurrente**
####Actividad Práctica (Opcional) - Comunicación y Sincronización

---

##**Ejercicio 2 - Babuinos en conflicto**

##Para compilar y ejecutar codigo en Java en Google Colab:

1.  **Instalar Java Development Kit (JDK)**

2.  **Guardar el codigo de Java como un archivo `.java`**

3.  **Compilar el codigo Java**: Usar el comando `javac`.

4.  **Correr el programa**: Usar el comando `java`.


In [ ]:
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
!java -version

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,533 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,013 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,149 kB]
Get:14 http://se

In [ ]:
%%writefile Babuinos.java
import java.util.concurrent.ThreadLocalRandom;
import java.util.concurrent.locks.Condition;
import java.util.concurrent.locks.ReentrantLock;

public class Babuinos {

    // Valores por defecto si no se pasan argumentos
    private static final int DEFAULT_EAST_COUNT = 12;
    private static final int DEFAULT_WEST_COUNT = 12;

    enum Direction {
        EAST_TO_WEST("Este -> Oeste"),
        WEST_TO_EAST("Oeste -> Este");

        private final String text;

        Direction(String text) {
            this.text = text;
        }

        Direction opposite() {
            return this == EAST_TO_WEST ? WEST_TO_EAST : EAST_TO_WEST;
        }

        int index() {
            return this == EAST_TO_WEST ? 0 : 1;
        }

        @Override
        public String toString() {
            return text;
        }
    }

    static class Rope {
        private static final int CAPACITY = 5;
        private static final int MAX_CONSECUTIVE = 5;
        private static final int NUM_DIRECTIONS = 2; // Para el tamaño del array 'waiting'

        private final ReentrantLock lock = new ReentrantLock(true);
        private final Condition canCross = lock.newCondition();

        private int onRope = 0;                 // babuinos colgados ahora
        private Direction currentDirection = null;   // null = cuerda libre
        private Direction lastDirection = null;
        private int consecutiveCrosses = 0;               // cruces seguidos en esa direccion
        private final int[] waiting = new int[NUM_DIRECTIONS]; // cola por direccion

        private int maxSimultaneous = 0;           // solo para estadisticas
        private int totalCrossings = 0;

        /** Se cuelga de la cuerda. Bloquea hasta que sea seguro hacerlo. */

        void climb(String name, Direction dir) throws InterruptedException {
            lock.lock();
            try {
                waiting[dir.index()]++;
                log(name, dir, "espera en la orilla (colgados: " + onRope + ")");

                while (!canClimb(dir)) {
                    canCross.await();
                }

                waiting[dir.index()]--;

                if (dir != lastDirection) {   // cambio de sentido
                    consecutiveCrosses = 0;
                    lastDirection = dir;
                }
                consecutiveCrosses++;
                onRope++;
                totalCrossings++;
                currentDirection = dir;
                maxSimultaneous = Math.max(maxSimultaneous, onRope);

                log(name, dir, "SE CUELGA de la cuerda (colgados: " + onRope + ")");
            } finally {
                lock.unlock();
            }
        }

        /**
         * Condicion de seguridad:
         *  - hay lugar en la cuerda, y
         *  - la cuerda esta vacia o ya se cruza en mi misma direccion, y
         *  - no abuse del turno mientras el otro lado espera.
         */
        private boolean canClimb(Direction dir) {
            if (onRope >= CAPACITY) {
                return false;
            }
            if (onRope > 0 && currentDirection != dir) {
                return false;
            }
            int effectiveConsecutive = (dir == lastDirection) ? consecutiveCrosses : 0;
            if (effectiveConsecutive >= MAX_CONSECUTIVE && waiting[dir.opposite().index()] > 0) {
                return false;   // cedo el paso al otro lado
            }
            return true;
        }

        /** Llego a la otra orilla y suelta la cuerda. */

        void descend(String name, Direction dir) {
            lock.lock();
            try {
                onRope--;
                if (onRope == 0) {
                    currentDirection = null;     // cuerda libre: puede cambiar el sentido
                }
                log(name, dir, "llego a la otra orilla (colgados: " + onRope + ")");
                canCross.signalAll();
            } finally {
                lock.unlock();
            }
        }

        void statistics() {
            System.out.println();
            System.out.println("== Resumen ==");
            System.out.println("Cruces realizados        : " + totalCrossings);
            System.out.println("Maximo simultaneo en la cuerda: " + maxSimultaneous
                    + " (limite " + CAPACITY + ")");
        }

        private void log(String name, Direction dir, String message) {
            System.out.printf("[%-10s] %-14s %s%n", name, dir, message);
        }
    }


    static class Baboon implements Runnable {
        private final String name;
        private final Direction direction;
        private final Rope rope;

        // Tiempos de espera en milisegundos
        private static final int MIN_ARRIVAL_TIME_MS = 50;
        private static final int MAX_ARRIVAL_TIME_MS = 400;
        private static final int MIN_CROSSING_TIME_MS = 300;
        private static final int MAX_CROSSING_TIME_MS = 900;

        Baboon(String name, Direction direction, Rope rope) {
            this.name = name;
            this.direction = direction;
            this.rope = rope;
        }

        @Override
        public void run() {
            try {
                // Tiempo variable hasta llegar al borde del cañon
                Thread.sleep(ThreadLocalRandom.current().nextInt(MIN_ARRIVAL_TIME_MS, MAX_ARRIVAL_TIME_MS));

                rope.climb(name, direction);
                try {
                    // Tiempo que tarda en cruzar colgado de la cuerda
                    Thread.sleep(ThreadLocalRandom.current().nextInt(MIN_CROSSING_TIME_MS, MAX_CROSSING_TIME_MS));
                } finally {
                    rope.descend(name, direction);
                }
            } catch (InterruptedException e) {
                Thread.currentThread().interrupt();
            }
        }
    }


    public static void main(String[] args) throws InterruptedException {

        int eastCount = args.length > 0 ? Integer.parseInt(args[0]) : DEFAULT_EAST_COUNT;
        int westCount = args.length > 1 ? Integer.parseInt(args[1]) : DEFAULT_WEST_COUNT;

        System.out.println("Babuinos que van Este -> Oeste: " + eastCount);
        System.out.println("Babuinos que van Oeste -> Este: " + westCount);
        System.out.println("Capacidad de la cuerda: " + Rope.CAPACITY + "\n"); // Usando la constante CAPACITY

        Rope rope = new Rope();
        Thread[] threads = new Thread[eastCount + westCount];
        int i = 0;

        for (int n = 1; n <= eastCount; n++, i++) {
            threads[i] = new Thread(new Baboon("E-" + n, Direction.EAST_TO_WEST, rope));
        }
        for (int n = 1; n <= westCount; n++, i++) {
            threads[i] = new Thread(new Baboon("O-" + n, Direction.WEST_TO_EAST, rope));
        }

        for (Thread t : threads) {
            t.start();
        }
        for (Thread t : threads) {
            t.join();
        }

        rope.statistics();
        System.out.println("Todos los babuinos cruzaron. Ninguno murio.");
    }
}


Overwriting Babuinos.java


In [ ]:
!javac Babuinos.java
!java Babuinos 12 12

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Babuinos que van Este -> Oeste: 12
Babuinos que van Oeste -> Este: 12
Capacidad de la cuerda: 5

[O-12      ] Oeste -> Este  espera en la orilla (colgados: 0)
[O-12      ] Oeste -> Este  SE CUELGA de la cuerda (colgados: 1)
[O-10      ] Oeste -> Este  espera en la orilla (colgados: 1)
[O-10      ] Oeste -> Este  SE CUELGA de la cuerda (colg

### Conclusiones
El desafío de este ejercicio radica en coordinar la concurrencia sobre un recurso compartido y de capacidad limitada (la cuerda). Para garantizar que no ocurran colisiones, fue necesario definir la cuerda como una región crítica gobernada por la mutua exclusión entre sentidos opuestos y regulada por un semáforo contador que restringe el aforo a cinco babuinos.
Este esquema de lectores-escritores, si bien permiti que múltiples babuinos crucen en la misma dirección maximiza el aprovechamiento del recurso y minimiza los cambios de sentido, una llegada continua de hilos en una dirección puede inducir starvation en la otra punta.
Asimismo, un diseño descuidado en la alternancia de turnos podría generar condiciones de espera circular conducentes a un deadlock.
Sincronizar el flujo mediante políticas de control que balanceen el caudal de cruce efectivo es clave.   